# 📓 Notebook 2｜最小分類管線：把 1.2 的四大問題走一遍

> 對應講義 **Part 2–3**（知識地圖站 2–3）
>
> 課本說分類系統有四階段：特徵**產生** → 特徵**選擇** → 分類器**設計** → 系統**評估**（見講義設計循環圖）。
> 本筆記本就是這四階段的「最小實作」：每一步都印出數字，讓你眼見為憑。

## 階段 1+2：產生與選擇特徵

沿用 nb1 的資料（兩類高斯）。「選擇」的判斷標準：在散點圖上**肉眼看分不分得開**——如果兩個特徵都用，當然最好；我們看看只用單一特徵會差多少。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
def gauss(n, mean, std): return mean + std * rng.standard_normal(n)

XA = np.stack([gauss(40, 3.0, 1.4), gauss(40, 3.0, 1.4)], axis=1)
XB = np.stack([gauss(40, 7.0, 1.4), gauss(40, 6.0, 1.4)], axis=1)
X = np.vstack([XA, XB]); y = np.array([0]*40 + [1]*40)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].scatter(X[:,0], X[:,1], c=y, cmap='bwr', s=30); axes[0].set_title('兩特徵都畫（2D）')
for i, ax in enumerate(axes[1:], 1):
    ax.scatter(X[:,i-1], np.zeros(len(y)) + i, c=y, cmap='bwr', s=30)
    ax.set_title(f'只用特徵 x{i}（1D）')
    ax.get_yaxis().set_visible(False)
plt.tight_layout(); plt.show()
print('兩個特徵一起用 → 2D 分離；單獨用 x1 或 x2 → 1D 也有部分分離（但會打架）')

## 階段 3：分類器設計（兩種方法）

### 方法一：手寫「暴力搜尋最佳決策線」（＝講義互動 Demo 1 的離線版）

In [ ]:
def line_accuracy(X, y, nx, ny, t0):
    '''決策線 nx*x + ny*y = t0；回傳準確率'''
    side = X @ np.array([nx, ny]) > t0          # 每個點在哪一側
    s0 = (y == 0).sum()                          # class 0 樣本數（用於決定哪側算 0）
    pred0 = (side[y == 0] == True).sum()
    pred1 = (side[y == 1] == False).sum()
    return (pred0 + pred1) / len(y)

best = {'acc': -1}
for deg in range(0, 180):                        # 搜尋 180 個方向
    th = np.deg2rad(deg); nx, ny = np.cos(th), np.sin(th)
    t = X @ np.array([nx, ny])
    ts = np.sort(t)
    for k in range(len(ts) - 1):                 # 最佳閾值必在相鄰投影點之間
        t0 = (ts[k] + ts[k+1]) / 2
        acc = line_accuracy(X, y, nx, ny, t0)
        if acc > best['acc']:
            best = {'acc': acc, 'nx': nx, 'ny': ny, 't0': t0}
print(f'暴力搜尋最佳線準確率 = {best["acc"]*100:.1f}%')
print(f'(法線方向 {np.degrees(np.arctan2(best["ny"], best["nx"])):.0f}°，閾值 {best["t0"]:.2f})')

# 畫圖
nx, ny, t0 = best['nx'], best['ny'], best['t0']
xs = np.linspace(0, 10, 100)
ys = (t0 - nx * xs) / ny
plt.figure(figsize=(6, 5))
plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', s=40)
plt.plot(xs, ys, 'g-', lw=2.5, label=f'最佳線（準確率 {best["acc"]*100:.1f}%）')
plt.xlim(0,10); plt.ylim(0,10); plt.legend(); plt.title('最佳決策線（暴力搜尋）')
plt.show()
print('注意：最佳也不是 100% —— 兩類重疊的部分是「隨機性」造成的，線再怎麼畫都無法全對。')

### 方法二：最近中心分類器（Nearest Centroid）

課本 Ch2 會教「最小距離/馬氏距離」；這裡先用最樸素版：**每類算平均當「代表點」，未知樣本交給最近的代表點**——這是分類器「設計」的最低門檻。

In [ ]:
centroid0 = X[y == 0].mean(axis=0)     # 良性類的平均特徵
centroid1 = X[y == 1].mean(axis=0)
print('class 0 中心:', centroid0.round(3), ' class 1 中心:', centroid1.round(3))

def nearest_centroid_predict(x):
    return 0 if np.linalg.norm(x - centroid0) < np.linalg.norm(x - centroid1) else 1

pred = np.array([nearest_centroid_predict(x) for x in X])
acc = (pred == y).mean()
print(f'最近中心分類器（自己在訓練資料上）準確率 = {acc*100:.1f}%')
# 混淆矩陣（課本「系統評估」的標準產物）
tn = ((pred == 0) & (y == 0)).sum(); fp = ((pred == 1) & (y == 0)).sum()
fn = ((pred == 0) & (y == 1)).sum(); tp = ((pred == 1) & (y == 1)).sum()
print('混淆矩陣：')
print(f'         預測良性  預測惡性')
print(f'實際良性   {tn:5d}   {fp:5d}')
print(f'實際惡性   {fn:5d}   {tp:5d}')

## 階段 4：系統評估 ——「樣本數 vs 準確率」實驗

課本強調：評估要用**沒看過的資料**（測試集）。下面的實驗：訓練樣本越多，最近中心分類器的測試準確率會怎麼變？

In [ ]:
def run_experiment(n_train, repeats=30):
    accs = []
    for _ in range(repeats):
        idx = rng.permutation(len(X))
        tr, te = idx[:n_train], idx[n_train:]
        c0 = X[tr][y[tr] == 0].mean(axis=0)
        c1 = X[tr][y[tr] == 1].mean(axis=0)
        pred = np.where(np.linalg.norm(X[te] - c0, axis=1) < np.linalg.norm(X[te] - c1, axis=1), 0, 1)
        accs.append((pred == y[te]).mean())
    return np.mean(accs)

ns = [4, 8, 12, 20, 30, 45, 60]
accs = [run_experiment(n) for n in ns]
plt.figure(figsize=(7, 4))
plt.plot(ns, [a*100 for a in accs], 'o-', c='#2563eb', lw=2)
plt.axhline(50, ls='--', c='#94a3b8', label='隨機猜測 50%')
plt.xlabel('訓練樣本數'); plt.ylabel('測試準確率 (%)')
plt.title('資料越多 → 分類器越穩（泛化的直覺）')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('樣本數最少時準確率最飄（且低）——這就是課本「樣本不足→效能差」的親身體驗。')

## 🏆 小結：你在 nb2 把「設計循環」走完了一遍
| 課本階段 | 你的實作 |
|---|---|
| 特徵產生 | 合成兩類高斯（mean/std 當特徵） |
| 特徵選擇 | 散點圖觀察 + 單/雙特徵比較 |
| 分類器設計 | 暴力搜尋最佳線 ・ 最近中心分類器 |
| 系統評估 | 準確率・混淆矩陣・樣本數實驗 |
| 回饋箭頭 | 看到「只用一個特徵分不開」→ 改回用兩個（就是回饋！） |

> 下一步：Ch2 貝氏分類器會告訴你「為什麼線性是最好的線」「怎麼算誤差機率」。